# Imports

In [1]:
import keras
import numpy as np
import pandas as pd
import transformers
import tensorflow as tf
from tqdm import tqdm
import matplotlib.pyplot as plt

I0000 00:00:1791796116.262725   70543 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791796116.311758   70543 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791796117.637531   70543 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except:
        pass

E0000 00:00:1791796120.437392   70543 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


# Dataset

Get the dataset from [here](https://tatoeba.org/en/downloads). Preferably use russian to english translations.

In [3]:
dataset = pd.read_csv('dataset.tsv', sep='\t', header=None, usecols=[1, 3])

In [4]:
dataset.head()

,1,3
0,Один раз в жизни я делаю хорошее дело... И оно...,For once in my life I'm doing a good deed... A...
1,Давайте что-нибудь попробуем!,Let's try something.
2,Мне пора идти спать.,I have to go to sleep.
3,Что ты делаешь?,What are you doing?
4,Что ты делаешь?,What do you make?


Use a custom tokenizer that can add bos and eos tokens (pass `add_special_tokens=True` when calling the tokenizer to add them).

In [5]:
class Tokenizer(transformers.GPT2Tokenizer):

    def build_inputs_with_special_tokens(self, token_ids_0, token_ids_1=None):
        if token_ids_1 is None:
            return [self.bos_token_id, *token_ids_0, self.eos_token_id]

        return [self.bos_token_id, *token_ids_0, self.bos_token_id, *token_ids_1, self.eos_token_id]

In [6]:
tokenizer = Tokenizer.from_pretrained('gpt2', cache_dir='../gpt2_tokenizer')
tokenizer.pad_token_id = tokenizer.eos_token_id

In [9]:
ru = [tokenizer.encode(i) for i in tqdm(dataset[1])]
en = [tokenizer.encode(i) for i in tqdm(dataset[3])]




































































































































































































100%|██████████| 830696/830696 [00:41<00:00, 20231.53it/s]


Since the dataset is rather large, you can omit the validation dataset and just use a set of test sentences after the training.

Create a dataset that returns the following
* A pair of tensors `((None, L), (None, P))` -- input sequence of tokens and output sequence of tokens to be fed into decoder (this should start with the BOS token)
* A tensor `(None, P)` -- output sequence of tokens to be predicted (this should end with EOS token)
* A tensor `(None, P)` -- a masking tensor marking padded tokens with 0

In [ ]:
class Dataset(keras.utils.PyDataset):
    def __init__(self, X, y, shuffle=False, batch_size=128, seed=1, **kwargs):
        super().__init__(**kwargs)
        self.X = X
        self.y = y
        self.shuffle = shuffle
        self.batch_size = batch_size
        self.rng = np.random.default_rng(seed)
        self.idx = np.arange(len(self.X))
        self.X_lens = np.array([len(e) for e in self.X])
        self.y_lens = np.array([len(e) for e in self.y])

        self.on_epoch_end()

    def __len__(self):
        return -(-len(self.X) // self.batch_size)

    def __getitem__(self, ind):
        i = self.idx[ind * self.batch_size: (ind + 1) * self.batch_size]

        max_x = np.max(self.X_lens[i])
        max_y = np.max(self.y_lens[i])

        x = keras.utils.pad_sequences(
            [self.X[e] for e in i], 
            maxlen=math.ceil(np.quantile(X_lens[i], self.q)),
            padding='post',
            truncating='post',
            value=tokenizer.pad_token_id
        )

        return self.X[i], self.y[i]

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.idx)


In [26]:
keras.utils.PyDataset?

Init signature:
keras.utils.PyDataset(
    workers=1,
    use_multiprocessing=False,
    max_queue_size=10,
)
Docstring:     
Base class for defining a parallel dataset using Python code.

Every `PyDataset` must implement the `__getitem__()` and the `__len__()`
methods. If you want to modify your dataset between epochs,
you may additionally implement `on_epoch_end()`,
or `on_epoch_begin` to be called at the start of each epoch.
The `__getitem__()` method should return a complete batch
(not a single sample), and the `__len__` method should return
the number of batches in the dataset (rather than the number of samples).

Args:
    workers: Number of workers to use in multithreading or
        multiprocessing.
    use_multiprocessing: Whether to use Python multiprocessing for
        parallelism. Setting this to `True` means that your
        dataset will be replicated in multiple forked processes.
        This is necessary to gain compute-level (rather than I/O level)
        benefits fr

# Model

Create a model for training. The model should have two inputs: input sequence `(None, L)` and output sequence`(None, P)`. The model output is a single tensor `(None, P)` logits (or probabilities) of the next token predicted for each input one.

In [ ]:
def get_model(
    units: int,
    n_tokens: int,
    n_labels: int,
    n_stacks: int = 1,
    bidirectional: bool = False,
    name: str | None = None,
    cell_type: type[keras.layers.Layer] = keras.layers.LSTMCell
) -> keras.Model:
    '''Creates a model with RNN architecture for sequence to sequence classification.

    Arguments:
        units: dimensionality of RNN cells
        n_tokens: number of tokens in the tokenizer dictionary
        n_labels: number of labels to be predicted
        n_stacks: number of RNN cells in the stack (1 -- no stacking)
        bidirectional: whether or not the model is bidirectional
        name: the model name
        cell_type: type of a cell to use, either keras.layers.LSTMCell or keras.layers.GRUCell

    Returns:
        The model'''
    ...

Try to add attention to your model (for example [additive attention](https://keras.io/api/layers/attention_layers/additive_attention/)), does it perform better?

# Training

Train your model using teacher forcing. The idea is that the model predicts the next token that should follow, so one part of the model (called encoder) reads the text and output some state containing information about the text read. The other part of the model (called decoder) reads an already generated text (or in case of the teacher forcing the expected output) and predicts the next token for each one. 

# Testing

Make a function for text translation. Translate some text and evaluate model performance.

Take note that your model is set for training. During the inference process you will have to use parts of the model independently (including the RNN cells).

In [ ]:
def translate(
    text: str,
    tokenizer: Tokenizer,
    model: keras.Model,
    max_len: int = 20
) -> str:
    '''Predicts `text`translation using the `model`.

    Arguments:
        text: text to be translated
        tokenizer: tokenizer to use
        model: model ot use
        max_len: maximum length of the prediction (in tokens)

    Returns:
        tranlated text'''
    ...